# Model Comparison — Inflation Forecasting

Loads saved test predictions from each model notebook and compares performance
on the **same held-out test period** using identical metrics.

**Run order (required before this notebook):**
1. `inflation_forecasting_elasticnet.ipynb` — saves `results/elasticnet_preds.npy`
2. `inflation_forecasting_rnn.ipynb` — saves `results/rnn_preds.npy`

All metrics are on the **transformed scale**: monthly log-diff × 100  
(percentage points of monthly PCE inflation). RMSE = 0.10 means forecasts  
are off by 0.10 pp per month on average.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

RESULTS_DIR = 'results'


## Load Saved Predictions

In [ ]:
def load_model_results(results_dir, model_key, dates_key=None, actuals_key=None):
    """Load predictions (and optionally dates/actuals) saved by a model notebook."""
    preds_path = os.path.join(results_dir, f'{model_key}_preds.npy')
    if not os.path.exists(preds_path):
        raise FileNotFoundError(
            f"Missing {preds_path}. Run the {model_key} notebook first."
        )
    preds = np.load(preds_path)
    dates = None
    actuals = None
    if dates_key:
        d_path = os.path.join(results_dir, f'{dates_key}_dates.npy')
        if os.path.exists(d_path):
            dates = pd.DatetimeIndex(np.load(d_path).astype('datetime64[ns]'))
    if actuals_key:
        a_path = os.path.join(results_dir, f'{actuals_key}_actuals.npy')
        if os.path.exists(a_path):
            actuals = np.load(a_path)
    return preds, dates, actuals


# ── Load ElasticNet results ───────────────────────────────────────────────────
en_preds, en_dates, _ = load_model_results(RESULTS_DIR, 'elasticnet', dates_key='elasticnet')
y_test = np.load(os.path.join(RESULTS_DIR, 'y_test.npy'))
y_test_dates = pd.DatetimeIndex(
    np.load(os.path.join(RESULTS_DIR, 'y_test_dates.npy')).astype('datetime64[ns]')
)

# ── Load RNN results ──────────────────────────────────────────────────────────
rnn_preds, rnn_dates, rnn_actuals = load_model_results(
    RESULTS_DIR, 'rnn', dates_key='rnn', actuals_key='rnn'
)

print(f"ElasticNet predictions : {len(en_preds)}  "
      f"({y_test_dates.min().date()} → {y_test_dates.max().date()})")
print(f"RNN predictions        : {len(rnn_preds)}  "
      f"({rnn_dates.min().date()} → {rnn_dates.max().date()})")


## Align Test Windows

The ElasticNet and RNN test sets may start on different dates: the RNN drops  
the first `SEQ_LEN - 1` test rows to build its initial context window.  
We compare only on the overlapping period.


In [ ]:
# Build date-indexed Series for each model
en_series  = pd.Series(en_preds,  index=y_test_dates, name='ElasticNet')
rnn_series = pd.Series(rnn_preds, index=rnn_dates,    name='LSTM')

# Actual values indexed by date
actual_series = pd.Series(y_test, index=y_test_dates, name='Actual')

# Align all on the RNN's (shorter) test window
common_dates = rnn_dates  # RNN window is always a subset of EN window

actual_aligned = actual_series.reindex(common_dates)
en_aligned     = en_series.reindex(common_dates)
rnn_aligned    = rnn_series.reindex(common_dates)

print(f"Comparison window: {common_dates.min().date()} to {common_dates.max().date()}")
print(f"Observations: {len(common_dates)}")


## Compare Models

In [ ]:
def compare_models(predictions_dict, y_true):
    """
    Compare model predictions against ground truth.

    Args:
        predictions_dict : dict mapping model name -> np.ndarray of predictions
        y_true           : np.ndarray of true values (same length as each prediction)

    Returns:
        pd.DataFrame sorted by RMSE ascending
    """
    y_true = np.asarray(y_true)
    rows = []
    for name, y_pred in predictions_dict.items():
        y_pred = np.asarray(y_pred)
        rows.append({
            'Model':    name,
            'RMSE':     float(np.sqrt(mean_squared_error(y_true, y_pred))),
            'MAE':      float(mean_absolute_error(y_true, y_pred)),
            'R²':  float(r2_score(y_true, y_pred)),
            'MAPE (%)': float(np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100),
        })
    return pd.DataFrame(rows).sort_values('RMSE').reset_index(drop=True)


results_df = compare_models(
    {
        'ElasticNet': en_aligned.values,
        'LSTM':       rnn_aligned.values,
        # Add more models here as they are trained:
        # 'GradientBoost': gb_aligned.values,
    },
    actual_aligned.values,
)

print(f"Test period : {common_dates.min().date()} to {common_dates.max().date()}")
print(f"n obs       : {len(common_dates)}")
print()
print("Metrics scale: monthly log-diff × 100 (pp of monthly PCE inflation)")
print()
print(results_df.to_string(index=False, float_format='{:.4f}'.format))


## RMSE Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(max(5, len(results_df) * 2), 4))
bars = ax.bar(results_df['Model'], results_df['RMSE'],
              color='steelblue', alpha=0.8, edgecolor='white')
ax.bar_label(bars, fmt='{:.4f}', padding=3, fontsize=10)
ax.set_ylabel('RMSE (monthly % points, lower is better)')
ax.set_title('Model Comparison: Out-of-Sample RMSE')
fig.tight_layout()
plt.show()


## Overlay Plot: Actuals vs All Models

In [ ]:
colors = ['steelblue', 'darkorange', 'seagreen', 'crimson', 'purple']

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(common_dates, actual_aligned.values,
        'o-', label='Actual', linewidth=2, markersize=4, color='black', alpha=0.7)

model_series = {
    'ElasticNet': en_aligned.values,
    'LSTM':       rnn_aligned.values,
}
for (name, preds), color in zip(model_series.items(), colors):
    ax.plot(common_dates, preds,
            's--', label=name, linewidth=1.6, markersize=4, color=color, alpha=0.85)

ax.axhline(0, color='grey', linewidth=0.5)
ax.set_title('Model Comparison: Predicted vs Actual PCEPI Monthly Growth')
ax.set_ylabel('PCEPI Monthly Growth (log diff × 100)')
ax.legend(loc='upper left', fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.tight_layout()
plt.show()


## Residuals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

model_series = {
    'ElasticNet': en_aligned.values,
    'LSTM':       rnn_aligned.values,
}
for ax, (name, preds) in zip(axes, model_series.items()):
    residuals = actual_aligned.values - preds
    ax.plot(common_dates, residuals, 'o-', linewidth=1.2, markersize=4, color='steelblue')
    ax.axhline(0, color='crimson', linewidth=1, linestyle='--')
    ax.set_title(f'{name} Residuals')
    ax.set_ylabel('Actual − Predicted (pp)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

fig.tight_layout()
plt.show()
